# Setup

In [35]:
# load required libraries with requirements.txt
! pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable


In [36]:
import matplotlib.pyplot as plt
import mysql.connector as mysql
import pandas as pd
import numpy as np
import json

from DbConnector import DbConnector

In [37]:
# Import datasets
credits_df = pd.read_csv("movies_ds/credits.csv")
keywords_df = pd.read_csv("movies_ds/keywords.csv")
links_small_df = pd.read_csv("movies_ds/links_small.csv")
links_df = pd.read_csv("movies_ds/links.csv")
movies_metadata_df = pd.read_csv("movies_ds/movies_metadata.csv")
ratings_small_df = pd.read_csv("movies_ds/ratings_small.csv")
ratings_df = pd.read_csv("movies_ds/ratings.csv")

/var/folders/vl/xlcg7z052993rf2f_9vpl4h00000gn/T/ipykernel_83081/1606664570.py:6: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  movies_metadata_df = pd.read_csv("movies_ds/movies_metadata.csv")


In [38]:
# Cleaning of the data
credits_df_clean = credits_df.copy()
keywords_df_clean = keywords_df.copy()
links_small_df_clean = links_small_df.copy()
links_df_clean = links_df.copy()
movies_metadata_df_clean = movies_metadata_df.copy()
ratings_small_df_clean = ratings_small_df.copy()
ratings_df_clean = ratings_df.copy()

# Getting rid of duplicates (movies with the same id)
movies_metadata_df_clean = movies_metadata_df_clean.drop_duplicates(subset='id')

# EDA

In [39]:
# Check rating range
print("\n--- Full Ratings Dataset ---")
print(f"Min rating: {ratings_df_clean['rating'].min()}")
print(f"Max rating: {ratings_df_clean['rating'].max()}")

# Unique keywords: some rows have empty lists, which is not null but still missing data
unique_keywords = keywords_df_clean['id'].nunique()
print(f"\nUnique keywords: {unique_keywords}")


--- Full Ratings Dataset ---
Min rating: 0.5
Max rating: 5.0

Unique keywords: 45432


In [40]:

print("Credits: \n", credits_df_clean.isnull().sum())
print("\nKeywords: \n", keywords_df_clean.isnull().sum())
print("\nLinks: \n", links_df_clean.isnull().sum())
print("\nRatings: \n", ratings_df_clean.isnull().sum())
print("\nMetadata: \n", movies_metadata_df_clean.isnull().sum())

Credits: 
 cast    0
crew    0
id      0
dtype: int64

Keywords: 
 id          0
keywords    0
dtype: int64

Links: 
 movieId      0
imdbId       0
tmdbId     219
dtype: int64

Ratings: 
 userId       0
movieId      0
rating       0
timestamp    0
dtype: int64

Metadata: 
 adult                        0
belongs_to_collection    40945
budget                       0
genres                       0
homepage                 37659
id                           0
imdb_id                     17
original_language           11
original_title               0
overview                   954
popularity                   5
poster_path                386
production_companies         3
production_countries         3
release_date                87
revenue                      6
runtime                    263
spoken_languages             6
status                      87
tagline                  25035
title                        6
video                        6
vote_average                 6
vote_count   

In [41]:
# Sample of movies_metadata_df_clean null values in title
print("\nLinks: \n")
print(movies_metadata_df_clean.loc[
    movies_metadata_df_clean["title"].isnull(),
    ["original_title", "title"],
])


Links: 

                                 original_title title
19729                              Midnight Man   NaN
19730  [{'iso_639_1': 'en', 'name': 'English'}]   NaN
29502                          マルドゥック・スクランブル 排気   NaN
29503      [{'iso_639_1': 'ja', 'name': '日本語'}]   NaN
35586                          Avalanche Sharks   NaN
35587  [{'iso_639_1': 'en', 'name': 'English'}]   NaN


In [42]:
bad_title = movies_metadata_df_clean[movies_metadata_df_clean["title"].isnull()]
print(bad_title[["id", "adult", "original_title", "release_date", "vote_count"]])

ids_numeric = pd.to_numeric(movies_metadata_df_clean["id"], errors="coerce")
print("Non-numeric ids:", movies_metadata_df_clean.loc[ids_numeric.isnull(), "id"].tolist())

# 'adult' should only be True/False
print(movies_metadata_df_clean["adult"].value_counts().head())

               id                                              adult  \
19729       82663                                              False   
19730  1997-08-20                                 - Written by Ørnås   
29502      122662                                              False   
29503  2012-09-29   Rune Balot goes to a casino connected to the ...   
35586      249260                                              False   
35587  2014-01-01   Avalanche Sharks tells the story of a bikini ...   

                                 original_title release_date  vote_count  
19729                              Midnight Man          NaN         NaN  
19730  [{'iso_639_1': 'en', 'name': 'English'}]            1         NaN  
29502                          マルドゥック・スクランブル 排気          NaN         NaN  
29503      [{'iso_639_1': 'ja', 'name': '日本語'}]           12         NaN  
35586                          Avalanche Sharks          NaN         NaN  
35587  [{'iso_639_1': 'en', 'name': 'English'

In [ ]:
# 1. Identify non-numeric IDs
ids_numeric = pd.to_numeric(movies_metadata_df_clean["id"], errors="coerce")
tail_idx = movies_metadata_df_clean.index[ids_numeric.isnull()]

if len(tail_idx) > 0:
    # The corresponding head rows immediately precede the tail rows
    head_idx = tail_idx - 1

    print(f"Total malformed rows to drop: {len(tail_idx)}")
    print("\nPreview of malformed rows:")
    print(
        movies_metadata_df_clean.loc[
            tail_idx, ["id", "original_title", "release_date"]
        ]
    )

    # Recover the good head rows:
    recovered_dates = movies_metadata_df_clean.loc[tail_idx, "id"].values
    movies_metadata_df_clean.loc[head_idx, "release_date"] = recovered_dates

    # Fix the title using original_title for the recovered rows
    movies_metadata_df_clean.loc[head_idx, "title"] = (
        movies_metadata_df_clean.loc[head_idx, "original_title"] 
    )
    
    # Drop malformed rows
    movies_metadata_df_clean = movies_metadata_df_clean.drop(index=tail_idx)
else:
    print("No non-numeric IDs found.")
    head_idx = pd.Index([])

# Fill any remaining missing titles with original_title
missing_title_idx = movies_metadata_df_clean[
    movies_metadata_df_clean["title"].isnull()
].index
if len(missing_title_idx) > 0:
    movies_metadata_df_clean.loc[missing_title_idx, "title"] = (
        movies_metadata_df_clean.loc[missing_title_idx, "original_title"]
    )

# 3. Ensure 'id' is numeric
movies_metadata_df_clean["id"] = pd.to_numeric(
    movies_metadata_df_clean["id"], errors="coerce"
)

# 4. Verification
print("\nVerification")
print(  "Null titles remaining:", movies_metadata_df_clean["title"].isnull().sum())
print("Non-numeric IDs remaining:", movies_metadata_df_clean["id"].isnull().sum(),)
print(f"Total remaining rows: {len(movies_metadata_df_clean)}")


all_fixed_idx = head_idx.union(missing_title_idx)
if len(all_fixed_idx) > 0:
    print("\nTitles Fixed")
    print(
        movies_metadata_df_clean.loc[
            all_fixed_idx, ["id", "original_title", "title", "release_date"]
        ]
    )

Total malformed rows to drop: 3

Preview of tail rows:
               id                            original_title release_date
19730  1997-08-20  [{'iso_639_1': 'en', 'name': 'English'}]            1
29503  2012-09-29      [{'iso_639_1': 'ja', 'name': '日本語'}]           12
35587  2014-01-01  [{'iso_639_1': 'en', 'name': 'English'}]           22

--- Verification ---
Null titles remaining: 0
Non-numeric IDs remaining: 0
Total remaining rows: 45433

--- Titles Fixed Preview ---
           id    original_title             title release_date
19729   82663      Midnight Man      Midnight Man   1997-08-20
29502  122662  マルドゥック・スクランブル 排気  マルドゥック・スクランブル 排気   2012-09-29
35586  249260  Avalanche Sharks  Avalanche Sharks   2014-01-01


In [45]:
# movies_metadata sample of ids
print("\nSample of movies_metadata_df ids:")
print(movies_metadata_df_clean.loc[head_idx, ["id", "imdb_id"]])

print("\nClean imdb_id in movies_metadata_df")
# Check how many start with 'tt'
sample_imdb = movies_metadata_df_clean['imdb_id'].dropna().astype(str)
starts_with_tt = sample_imdb.str.startswith('tt').all()
print(f"All non-null imdb_id start with 'tt': {starts_with_tt}")

# Remove 'tt' prefix and convert remainder to nullable integer
movies_metadata_df_clean['imdb_id_clean'] = movies_metadata_df_clean['imdb_id'].astype(str).str.replace('tt', '', regex=False)
movies_metadata_df_clean['imdb_id_clean'] = pd.to_numeric(movies_metadata_df_clean['imdb_id_clean'], errors='coerce').astype('Int64')

print("Sample cleaned imdb_id values:")
print(movies_metadata_df_clean[['imdb_id', 'imdb_id_clean']].head(3))

# Drop the old imdb_id column and rename the clean version
movies_metadata_df_clean = movies_metadata_df_clean.drop(columns=['imdb_id'])
movies_metadata_df_clean = movies_metadata_df_clean.rename(columns={'imdb_id_clean': 'imdb_id'})



Sample of movies_metadata_df ids:
           id    imdb_id
19729   82663  tt0113002
29502  122662  tt2423504
35586  249260  tt2622826

Clean imdb_id in movies_metadata_df
All non-null imdb_id start with 'tt': True
Sample cleaned imdb_id values:
     imdb_id  imdb_id_clean
0  tt0114709         114709
1  tt0113497         113497
2  tt0113228         113228
